# Lab 6 — Beyond self_debug: escalation + multi-step deep_research

**Lab 5** built `self_debug` — one `Step` → working, executed code, with two
gates (`execution_evaluator` + `step_evaluator`) and bounded retries. That
loop is great when the engineer just needs more attempts. But two whole
classes of problem stay out of reach for it:

1. **Failures the strict loop *structurally cannot* fix** — a missing package
   (the engineer prompt forbids it from installing things), or a renamed /
   removed API (the docs aren't necessarily in training data).
2. **Multi-step plans** — Lab 4 produces a `Plan` of `Step`s, and Lab 5 runs
   *one* step. Something has to run the whole plan, *carrying context between
   steps* so step N can use step N-1's outputs (`from step_1 import …`, load
   `data/foo.csv`, …).

Lab 6 adds the two extensions:

| Module | What it does | Status |
| --- | --- | --- |
| **`escalation`** | One bounded Claude Agent SDK call as an escape hatch when `failure_kind ∈ {missing_module, renamed_api}`. | new (opt-in) |
| **`deep_research`** | Outer orchestrator that iterates a `Plan` through `self_debug`, threading per-step summaries + a workspace file manifest into each next engineer. | new |

By the end of this lab you'll be able to:

1. Explain when escalation triggers, what bounds it, and the one trade-off
   (escalation = the only Anthropic-coupled piece of `cmbagent_lg`).
2. Watch a `missing_module` or `renamed_api` failure get rescued by a Claude
   SDK agent that web-searches and patches.
3. Run a whole plan end-to-end through `deep_research`, and see how step N
   sees step N-1's code, stdout, and produced files.
4. Inspect the artifact tree a multi-step run leaves on disk
   (`codebase/` + `data/` + `logs/`).

## Prerequisites

Same setup as Lab 5, plus:

- **For escalation**: `pip install -e '~/GitHub/cmbagent_lg[escalation]'` (the
  `[escalation]` extra pulls in `claude-agent-sdk`). And either set
  `ANTHROPIC_API_KEY` in `~/GitHub/cmbagent_lg/.env`, or rely on the ambient
  `claude` CLI's existing login (the SDK shells out to it).

- **For the demos**: `pip install numpy scipy matplotlib pandas`. The MLP
  demo also uses `scikit-learn` — but you can leave that out; if missing
  it'll trigger an escalation that installs it for you (combined demo
  bonus!).

Pick the **Python 3.12 (cmbagent_lg)** kernel for this notebook.

In [ ]:
from dotenv import load_dotenv

load_dotenv('/Users/boris/GitHub/cmbagent_lg/.env', override=True)

import os
assert os.environ.get('GOOGLE_API_KEY'), 'missing GOOGLE_API_KEY in .env'

# Langfuse is optional — attach the handler if its keys are present.
callbacks, handler = [], None
try:
    from cmbagent_lg.tracing import langfuse_handler
    handler = langfuse_handler()
    callbacks = [handler]
    print('env OK — Langfuse tracing attached')
except Exception as e:
    print(f'env OK — running without Langfuse ({e})')

## 1. Escalation — the escape hatch

`self_debug`'s strict loop has two structural blind spots:

| Blind spot | Why the strict loop can't | What escalation does |
| --- | --- | --- |
| **Missing package** | `engineer.yaml` explicitly forbids the engineer from installing — that's "a separate concern". | `pip install <pkg>` via `Bash`, then re-run the same code. |
| **Renamed / removed API** | The engineer model may not know about the change (post-cutoff, niche, version-pinned). The error message often doesn't reveal the fix. | `WebFetch` the package's migration guide / changelog, then make a **minimal `Edit`** to the script. |

Escalation is **opt-in** (`PlanContext.enable_escalation = True`), **one-shot
per step** (a `state["escalated"]` flag prevents loops), and **bounded** by
budget + turn caps. It does NOT consume an engineer attempt.

### Classifying the failure

The decision lives in **`execution_evaluator`**'s structured output. The
`ExecutionVerdict` schema gained a `failure_kind` field:

```python
class ExecutionVerdict(BaseModel):
    status: Literal["success", "failure"]
    failure_kind: Optional[Literal["missing_module", "renamed_api", "other"]]
    error_summary: Optional[str]
    fix_suggestion: Optional[str]
```

The router (`route_after_execution_evaluator`) escalates when
`failure_kind ∈ {"missing_module", "renamed_api"}` and we haven't escalated
this step yet. *Other* failures stay on the strict-engineer path.

Why the LLM classifies and not a stderr regex? Because the engineer prompt
encourages catching exceptions and printing them — which means the error
text lands in **stdout**, not as a traceback in stderr. A regex on stderr
misses those entirely; an LLM that reads both does not.

### The flow

`self_debug` now has the third branch — only fires when `enable_escalation`
is True and the failure is classified as escalatable.

In [ ]:
from cmbagent_lg import self_debug_graph
from IPython.display import Image, display

try:
    display(Image(self_debug_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(self_debug_graph.get_graph().draw_ascii())

### The bounds

The escalation node builds a `ClaudeAgentOptions` from these `PlanContext`
knobs — pick a cheap model and a tight budget for routine cases:

| Knob | What it does |
| --- | --- |
| `enable_escalation` | Off by default. Set True to opt in. |
| `escalation_model` | Which Claude. Default = SDK default (Sonnet-class). Set to `"claude-haiku-4-5-20251001"` for ~5–10× cheaper runs that easily handle "look at a changelog and edit one line". |
| `escalation_max_budget_usd` | Hard $ ceiling. The SDK stops when the estimate hits it. |
| `escalation_max_turns` | Cap on agentic turns. |

**One honest trade-off:** the rest of `cmbagent_lg` is model-agnostic
(swap Gemini for any LangChain provider in `llms.py`). `escalation` is the
single place that hard-couples you to Anthropic — `claude-agent-sdk` only
talks to Claude. The plan calls this "Path A". A "Path B" — a native
LangGraph ReAct subagent on your own model — exists in design but isn't
built; pick this trade-off knowingly.

### Demo — a removed API the engineer can't escape

NumPy 2.0 renamed `np.trapz` → `np.trapezoid` and later removed `np.trapz`.
Modern Gemini sometimes knows this, sometimes doesn't — making it a flaky
test on its own. So the Step *forces* `numpy.trapz` ("hard requirement,
do not substitute") to reliably trigger a `renamed_api` failure. Then
escalation web-fetches the migration guide and patches the call.

(Run takes ~30–60s, costs a few cents of Haiku.)

In [ ]:
from cmbagent_lg import PlanContext, Step, self_debug_graph, prepare_work_dir

ctx = PlanContext(
    main_task='Numerically integrate a function with the trapezoidal rule.',
    code_execution_timeout=30,
    max_n_attempts=3,
    enable_escalation=True,
    escalation_model='claude-haiku-4-5-20251001',
    escalation_max_budget_usd=0.50,
    escalation_max_turns=12,
)

step = Step(
    sub_task='Numerically integrate sin(x) on [0, pi] with the trapezoidal rule.',
    sub_task_agent='engineer',
    bullet_points=[
        'Build an array of 1000 evenly spaced x values from 0 to pi.',
        'Evaluate y = sin(x) on that grid.',
        # Forces the engineer to write the broken API:
        'Integrate y over x by calling numpy.trapz(y, x) — use that exact function. '
        'Do NOT substitute scipy, np.trapezoid, or a manual implementation; '
        'numpy.trapz is a hard requirement.',
        'Print the integral value (the exact answer is 2.0).',
    ],
    code_execution_timeout=30,
)

work_dir = prepare_work_dir('work_dir/lab6_escalation')

result = self_debug_graph.invoke(
    {'step': step, 'work_dir': str(work_dir), 'step_number': 1},
    context=ctx,
    config={'callbacks': callbacks},
)

print(f'attempts:   {result["attempts"]} / {ctx.max_n_attempts}')
print(f'escalated:  {result["escalated"]}')
print(f'exec verdict: {result["current_execution_verdict"].status}')
print(f'step verdict: fulfilled={result["current_step_verdict"].fulfilled}')

### What did the escalation actually do?

After the run, `logs/step_1_escalation.json` records exactly what happened
inside the SDK call — cost, turns, tool calls, whether it edited the code.

In [ ]:
import json
rec = json.loads((work_dir / 'logs' / 'step_1_escalation.json').read_text())
for k, v in rec.items():
    print(f'  {k:14s} {v}')

And the audit trail is on disk: the broken first attempt is preserved
as `step_1_failure_1.py`, the fixed version is `step_1.py`. Let's see the
single-line patch:

In [ ]:
import subprocess
print('--- diff: failure_1 vs the escalation-fixed version ---')
print(subprocess.run(
    ['diff', '-u',
     str(work_dir / 'codebase' / 'step_1_failure_1.py'),
     str(work_dir / 'codebase' / 'step_1.py')],
    capture_output=True, text=True,
).stdout)

## 2. Multi-step deep_research

Planning (Lab 4) produces a `Plan` of `Step`s. self_debug (Lab 5) runs *one*
Step. The missing piece — and the one cmbagent itself solves with the
deep-research workflow — is the **outer orchestrator** that runs the whole
plan, step by step, *carrying context between steps* so step N can use what
step N-1 wrote.

A naive `for step in plan: self_debug(step)` doesn't work: each next
engineer starts fresh, knows nothing about prior code, files, or stdout. It
can't `from step_1 import …` or load `data/primes.csv` if it doesn't know
those exist.

### The carryover, cmbagent-style

cmbagent's deep-research workflow accumulates a `previous_steps_execution_summary`
string — one block per completed step (header + executed code + stdout) plus
a freshly-scanned workspace file manifest (paths in `codebase/`, `data/`,
`logs/`). That string is injected into each next engineer's system prompt.

`cmbagent_lg/deep_research/` does the same thing as a tiny LangGraph
subgraph:

- **`DeepResearchState`** — `plan`, `step_index`, `step_summaries`,
  `step_outcomes`, `work_dir`.
- **`run_step`** node — builds the summary block + a fresh manifest scan,
  invokes `self_debug_graph` for the current `Step`, threading the summary
  through via a new `DebugState["previous_steps_execution_summary"]` field
  (which the engineer node renders into its prompt as
  `{previous_steps_execution_summary}`).
- **`after_step`** router — `END` if step failed or plan exhausted; else
  next step.

Per-step state reset (`attempts`, `error_history`, `escalated`, …) is
automatic — each `self_debug_graph.invoke(...)` is a fresh run.

In [ ]:
from cmbagent_lg import deep_research_graph

try:
    display(Image(deep_research_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(deep_research_graph.get_graph().draw_ascii())

### The cross-step I/O rules

A subtle gotcha caught by an early demo: the engineer at step 2 saved data
via `np.savez(history_path, train=..., val=...)`; step 3 then guessed
`history['train_losses']` — pattern-matching on a *variable name* it saw in
step 2's code, not the actual npz key. Even worse, the relevant `np.savez`
line was past the head+tail truncation budget of the carryover summary.

Two prompt-level fixes (now in `engineer.yaml`) prevent the whole class:

- **Producer** — when saving a structured artifact (`.npz`, `.parquet`,
  `.pkl`, multi-column `.csv`, …), **print its schema** right after:
  `print("saved data/X.npz keys:", list(np.load("data/X.npz").files))`.
- **Consumer** — when loading a previous step's file, **inspect first**
  (`list(np.load(p).files)`), don't infer schema from variable names you
  saw in earlier code.

Plus: the carryover summary now ships **full code, no truncation** — the
deep-research orchestrator passes step code through `_build_step_summary`
without `_head_tail`, so the next engineer sees every line.

### v1 scope (what's deferred)

- **Engineer-only steps.** A non-engineer step (e.g. `researcher`) halts
  the plan cleanly with a clear outcome. Set
  `available_agents=[("engineer", "…")]` on the planning context to force
  engineer-only plans.
- **Halt on first failure.** No deep_research-level retry — `self_debug`'s
  bounded `max_n_attempts` is the inner budget; if a step exhausts it, the
  plan stops.

### Demo — generate, train, evaluate a 5D MLP

A vague 4-sentence main_task is enough — the planner decomposes it. With
`enable_escalation=True`, a missing `scikit-learn` (or `torch`) trips an
escalation that pip-installs it mid-run. The whole pipeline — planning →
deep_research → per-step self_debug → optional escalation — happens in one
graph invocation.

(Run takes ~60–120s. Escalation cost depends on which packages are missing
in your venv; typically $0.05–0.20 if any.)

In [ ]:
from cmbagent_lg import (
    PlanContext, deep_research_graph,
    graph as planning_graph, save_final_plan, save_deep_research_summary,
)

ctx = PlanContext(
    main_task=(
        'Investigate how well a small MLP can learn a smooth nonlinear '
        'function of five random inputs. Generate a synthetic 5D regression '
        'dataset, train an MLP on it, and produce diagnostic plots of the '
        'result.'
    ),
    code_execution_timeout=120,
    max_n_attempts=3,
    available_agents=[
        ('engineer', 'Writes and runs Python code: numeric work, data I/O, plotting, ML.'),
    ],
    maximum_number_of_steps_in_plan=3,
    num_rounds=1,
    enable_escalation=True,
    escalation_model='claude-haiku-4-5-20251001',
)

work_dir = prepare_work_dir('work_dir/lab6_deep_research')

# Phase 1 — produce the Plan.
plan = planning_graph.invoke({}, context=ctx, config={'callbacks': callbacks})['current_plan']
save_final_plan(plan, work_dir)
print(plan.format())

In [ ]:
# Phase 2 — execute the Plan step by step.
dr_result = deep_research_graph.invoke(
    {'plan': plan, 'work_dir': str(work_dir)},
    context=ctx,
    config={'callbacks': callbacks},
)

outcomes = dr_result['step_outcomes']
print('=== STEP OUTCOMES ===')
for o in outcomes:
    flag = '✓' if o.get('fulfilled') else '✗'
    extras = [f'attempts={o.get("attempts", "-")}']
    if o.get('escalated'): extras.append('escalated')
    if not o.get('fulfilled') and o.get('reason'): extras.append(f'reason={o["reason"]!r}')
    print(f'  {flag} step {o["step_number"]}  ' + ' '.join(extras))

all_ok = all(o.get('fulfilled') for o in outcomes)
print(f'=== PLAN: {"COMPLETE" if all_ok else "HALTED"} '
      f'({len(outcomes)}/{len(plan.sub_tasks)} steps) ===')

save_deep_research_summary(work_dir, plan, outcomes, dr_result.get('step_summaries', []))

### Inspecting what landed on disk

`deep_research` writes a predictable tree — cmbagent-style:

```
work_dir/
├── codebase/   step_N.py (+ step_N_failure_I.py audit trail) + .log
├── data/       output files the engineers wrote (relative path "data/...")
└── logs/       step_N_{execution_verdict,verdict,data_manifest,
                 timings,escalation}.json + deep_research_run.json
```

In [ ]:
import os
from pathlib import Path
for root, _, files in sorted(os.walk(work_dir)):
    rel = Path(root).relative_to(work_dir)
    for f in sorted(files):
        print(f'  {rel / f if str(rel) != "." else f}')

### Did the carryover actually work?

If the demo produced more than one step, the later steps should reference
the earlier ones' outputs directly. Quick check:

In [ ]:
# Show any cross-step references in later step scripts.
import re
codebase = work_dir / 'codebase'
for p in sorted(codebase.glob('step_*.py')):
    if 'failure' in p.name: continue
    refs = [m.group(0) for m in re.finditer(r'data/[\w._-]+', p.read_text())]
    if refs:
        print(f'  {p.name:14s} references: {sorted(set(refs))}')

## 3. The full picture

You now have four composable modules, each a small LangGraph subgraph:

| Module | One-line role |
| --- | --- |
| **`planning`** (Lab 4) | research task → structured `Plan` of `Step`s |
| **`self_debug`** (Lab 5) | one `Step` → working executed code, two gates, bounded retries |
| **`escalation`** (Lab 6 §1) | escape hatch for `missing_module` / `renamed_api`, bounded Claude SDK call |
| **`deep_research`** (Lab 6 §2) | runs a whole `Plan` step by step, carrying context |

Every LLM call across all four is tagged with its agent role, so the
existing `cmbagent-lg-cost` CLI breaks the run down per agent — Gemini for
planning/self_debug nodes, Claude only where escalation actually fired.

In [ ]:
import sys, time, subprocess
from langfuse import get_client

if handler is not None and handler.last_trace_id:
    trace_id = handler.last_trace_id
    print(f'latest trace: http://localhost:3000/trace/{trace_id}\n')
    get_client().flush()
    time.sleep(3)
    out = subprocess.run(
        [sys.executable, '-m', 'cmbagent_lg.cli', 'cost', trace_id],
        capture_output=True, text=True,
    )
    print(out.stdout or out.stderr)
else:
    print('Langfuse not attached — skip this cell, or set LANGFUSE_* in .env.')

## Recap

**Concepts**

- **Escalation** is the *opt-in escape hatch* for failures the strict loop
  structurally can't fix: `missing_module` (engineer can't install) and
  `renamed_api` (model may not know the new path). The
  `execution_evaluator`'s LLM-classified `failure_kind` is what gates it,
  not a stderr regex (caught-and-printed errors must still be classified).
  It's bounded by budget + turn caps, runs one-shot per step, and is the
  one Anthropic-coupled piece of the package.
- **`deep_research`** is the outer orchestrator: it iterates a `Plan` of
  `Step`s through `self_debug`, threading prior steps' code + stdout + a
  workspace file manifest into the next engineer's prompt as
  `{previous_steps_execution_summary}`. Per-step state reset is automatic;
  the only cross-step state is the `step_summaries` accumulator and the
  outcomes list.
- **Cross-step I/O rules** in the engineer prompt (print schemas on save,
  inspect on load) keep step N from guessing at schemas it can't see.

**Code**

- `enable_escalation=True` + `escalation_model='claude-haiku-4-5-…'` on
  `PlanContext` to turn the escape hatch on.
- `deep_research_graph.invoke({'plan': plan, 'work_dir': ...}, context=ctx,
  config=...)` after a planner run produces the plan.
- `logs/step_N_escalation.json` records every escalation; `logs/deep_research_run.json`
  records the whole run's outcomes.

**Where this is going**

- A richer `step_evaluator` that inspects file *contents* (CSV columns,
  array shapes) — not just paths/sizes — plus VLM-based plot review.
- Specialist routing on the escalation path (a dedicated `installer`, a
  library-context agent for CAMB / cobaya).
- Persistence: resumable runs via per-step state snapshots
  (cmbagent's `context_step_N.pkl` pattern).